# RAG Fusion — Multiple Queries, One Better Answer
## Rewrite, Rank, Fuse — Beating Single-Query Retrieval
⏱ ~15 min

**RAG Fusion** solves the single biggest weakness of standard RAG: *a single query is a single point of failure.* If your phrasing doesn't land in the right region of embedding space, you miss relevant documents entirely — and the LLM answers from an incomplete picture.

RAG Fusion (Lawrence 2024) fixes this at the retrieval level:
1. **Fan out** — generate N semantically diverse paraphrases of the original query
2. **Retrieve** — run each paraphrase through the vector store independently and in parallel
3. **Fuse** — merge all ranked result lists using Reciprocal Rank Fusion (RRF)
4. **Generate** — answer from the deduplicated, fused context

By the end of this workshop you will understand *why* single-query RAG fails, *how* RRF combines ranked lists without score normalisation, and *how* the LangGraph Send API lets you fan out N retrieval branches in true parallel.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why single-query RAG fails and what fusion adds |
| 2 | **Setup** — Knowledge base, embeddings, and retriever |
| 3 | **Query Generation** — Fan-out with the LangGraph Send API |
| 4 | **RRF Scoring** — Merging ranked lists by inverse rank |
| 5 | **Full Pipeline** — End-to-end RAG Fusion graph |
| 6 | **Experiments** — Ablations: k, NUM_VARIANTS, RRF_K |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+
- `OPENAI_API_KEY` in `.env` (local) or Colab Secrets
- Familiarity with basic RAG (see example 12 first if needed)

### Key References
> Cormack, G. V., Clarke, C. L. A., & Buettcher, S. (2009). *Reciprocal Rank Fusion outperforms Condorcet and individual Rank Learning Methods.* SIGIR 2009. https://dl.acm.org/doi/10.1145/1571941.1572114
>
> Lewis, P., Perez, E., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401
>
> Ma, X., et al. (2023). *Query Rewriting for Retrieval-Augmented Large Language Models.* https://arxiv.org/abs/2305.14283
>
> Lawrence, S. (2024). *RAG Fusion.* https://github.com/Raudaschl/rag-fusion

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "langchain",
            "langchain-openai",
            "langchain-chroma",
            "langchain-community",
            "chromadb",
            "langgraph",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ────────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon), name = OPENAI_API_KEY
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

# ── Sanity check ───────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY not found or malformed.\n"
    "  Local: add OPENAI_API_KEY=sk-... to your .env file\n"
    "  Colab:  Secrets panel -> Add secret 'OPENAI_API_KEY'"
)
print(f"OPENAI_API_KEY present: True (starts with sk-...)")

---

## Part 1 — Why Single-Query RAG Fails

---

### The Single-Query Problem

Standard RAG converts one user query into one embedding vector, then finds the k nearest chunks. This works well when the user's wording happens to align with how the documents were phrased. It fails when there is a vocabulary mismatch.

**Example:** A document says *"adverse reactions to penicillin include anaphylaxis."* A user asks *"is amoxicillin safe if I'm allergic to antibiotics?"* — conceptually related, but the embeddings may not land close enough to retrieve that chunk.

### Three Strategies Compared

| Strategy | What it does | Coverage | Cost | Latency |
|----------|-------------|----------|------|---------|
| **Standard RAG** | Single query, top-k retrieval | One embedding neighbourhood | Low | Low |
| **HyDE** | LLM writes a fake answer, embed that | One hypothetical doc space | Medium | Medium |
| **RAG Fusion** | N query paraphrases, N retrievals, RRF merge | Union of N embedding neighbourhoods | Medium | Medium (parallel) |

RAG Fusion and HyDE are complementary. Fusion improves *recall breadth* by covering more of the embedding space. HyDE improves *embedding alignment* by matching document style. They can be combined.

### Query Decomposition Strategies

| Strategy | How | When to use |
|----------|-----|-------------|
| **Paraphrasing** | Same meaning, different words | Vocabulary mismatch |
| **Perspective shifting** | "What would a doctor ask? A patient?" | Multi-stakeholder queries |
| **Decomposition** | Split compound question into sub-questions | Multi-hop reasoning |
| **Abstraction** | Move up or down the specificity ladder | Narrow vs. broad coverage |

RAG Fusion typically uses **paraphrasing** — same intent, different surface form — to widen retrieval coverage without changing what the system is looking for.

---

### RAG Fusion Architecture

```
User query
    |
    v
+-------------------------+
| LLM: Multi-Query        |  generates N paraphrases
| Generator               |  of the original question
+----------+--------------+
           |
     +-----+-----+
     |           |  (LangGraph Send API -- parallel)
     v           v
  [q1]         [q2]         [q3]         [q4]
   |             |             |             |
   v             v             v             v
Vector         Vector        Vector        Vector
Store          Store         Store         Store
top-k          top-k         top-k         top-k
   |             |             |             |
   +-------------+-------------+-------------+
                         |
                         v
            +------------------------+
            | Reciprocal Rank        |  score = sum(1 / (k + rank_i))
            | Fusion (RRF)           |  across all ranked lists
            +------------+-----------+
                         |
                         v
              Ranked, deduplicated docs
                         |
                         v
                +----------------+
                |      LLM       |  answers from fused context
                +--------+-------+
                         |
                         v
                      Answer
```

> **Key insight:** The vector store is queried N times, but in *parallel* — total latency is roughly the same as a single retrieval call, not N times single-retrieval.

---

## Part 2 — Setup: Knowledge Base, Embeddings, and Retriever

---

We build a small in-memory knowledge base about LLM tooling. In a real system this would be loaded from PDFs, databases, or a document store — the retrieval pattern is identical.

### Key Constants

| Constant | Value | Meaning |
|----------|-------|---------|
| `NUM_VARIANTS` | 4 | Number of query paraphrases to generate |
| `K` | 3 | Top-k documents retrieved per variant |
| `RRF_K` | 60 | RRF smoothing constant (Cormack et al. 2009 default) |

`RRF_K = 60` was chosen in the original paper to reduce the influence of high-variance top-1 rankings. A smaller value (e.g. 10) amplifies rank-1 documents; a larger value (e.g. 100) flattens rank differences.

In [ ]:
# ─── 2-A: Build the knowledge base ───────────────────────────────────────────
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# ── RAG Fusion constants ───────────────────────────────────────────────────────
NUM_VARIANTS = 4   # query paraphrases to fan out
K = 3              # top-k docs per variant retrieval
RRF_K = 60         # RRF constant (standard value from Cormack et al. 2009)

DOCS = [
    Document(page_content="LangGraph is a library for building stateful, multi-actor LLM applications using graph-based workflows."),
    Document(page_content="LangChain provides tools for building LLM applications: chains, agents, and retrieval systems."),
    Document(page_content="Vector databases store embeddings for semantic search. Popular options: Chroma, Pinecone, Weaviate."),
    Document(page_content="RAG (Retrieval-Augmented Generation) grounds LLM responses in external documents to reduce hallucination."),
    Document(page_content="Reciprocal Rank Fusion (RRF) merges ranked lists: score = sum(1 / (k + rank)). Lower rank = higher score."),
    Document(page_content="The LangGraph Send API allows parallel node execution by dispatching the same event to multiple workers."),
    Document(page_content="Query expansion generates multiple paraphrases of a question to improve recall across embedding representations."),
    Document(page_content="BM25 ranks documents by term frequency and inverse document frequency — keyword-based retrieval."),
    Document(page_content="Hybrid search combines vector similarity and BM25, merging results with ensemble or RRF methods."),
    Document(page_content="Self-RAG grades retrieved documents and uses reflection tokens to decide when retrieval is needed."),
]

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    DOCS,
    embeddings_model,
    collection_name="rag_fusion_workshop",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": K})

print(f"Knowledge base ready: {len(DOCS)} documents")
print(f"Constants: NUM_VARIANTS={NUM_VARIANTS}, K={K}, RRF_K={RRF_K}")

In [ ]:
# ─── 2-B: Baseline -- single-query retrieval (what RAG Fusion improves on) ────
# Always start by understanding the baseline before adding complexity.

baseline_query = "How does RAG Fusion work?"
baseline_results = retriever.invoke(baseline_query)

print(f"Single-query retrieval for: '{baseline_query}'")
print(f"Retrieved {len(baseline_results)} chunks:\n")
for i, doc in enumerate(baseline_results, 1):
    print(f"  [{i}] {doc.page_content}")

print()
print("Observation: only chunks that match THIS exact phrasing are returned.")
print("A user asking 'what is multi-query retrieval?' might get different results.")

In [ ]:
# ─── 2-C: Demonstrate the vocabulary mismatch problem ─────────────────────────
# Same concept, different phrasing -> different retrieved chunks.

equivalent_queries = [
    "How does RAG Fusion work?",
    "What is multi-query retrieval with rank merging?",
    "Explain combining multiple search results with reciprocal rank fusion",
    "Why generate query variants before searching a vector store?",
]

print("Same concept, four phrasings -- note how retrieved sets differ:\n")
all_retrieved = []
for query in equivalent_queries:
    docs = retriever.invoke(query)
    snippets = [d.page_content[:60] + "..." for d in docs]
    all_retrieved.extend(d.page_content for d in docs)
    label = (query[:55] + "...") if len(query) > 55 else query
    print(f"  '{label}'")
    for s in snippets:
        print(f"      -> {s}")
    print()

unique_docs = len(set(all_retrieved))
print(f"Union of all retrieved docs: {unique_docs} unique chunks (vs {K} per single query)")
print("RAG Fusion recovers ALL of these -- then RRF ranks them by cross-query agreement.")

---

## Part 3 — Query Generation and the LangGraph Send API

---

### The LangGraph Send API

Standard LangGraph edges move from one node to the next sequentially. The **Send API** enables true fan-out: you return a list of `Send(node_name, state)` objects from an edge function, and LangGraph dispatches them all in parallel.

```
Normal edge:
  generate_variants ───────────────────> retrieve_variant
                         (sequential)

Send API (conditional edges returning Send objects):
  generate_variants --- Send(v1) ---> retrieve_variant
                    --- Send(v2) ---> retrieve_variant  (all parallel)
                    --- Send(v3) ---> retrieve_variant
                    --- Send(v4) ---> retrieve_variant
```

The results from all parallel workers are **merged** back into the shared state. This requires the receiving field to use `Annotated[list, operator.add]` — each worker's return value is appended to the list rather than overwriting it.

### State Design

```python
class RAGFusionState(TypedDict):
    question:    str              # original user query
    variants:    list[str]        # N paraphrases (set by generate_variants)
    ranked_docs: Annotated[list[list[str]], add]  # each worker appends one ranked list
    fused_docs:  list[str]        # RRF output (set by fuse)
    answer:      str              # final LLM answer (set by generate)
```

The `Annotated[list[list[str]], add]` type is what makes parallel merging work. Without it, each parallel worker would overwrite the shared field and you would only see the last result.

In [ ]:
# ─── 3-A: Generate query variants with an LLM ─────────────────────────────────
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def generate_query_variants(question: str, n: int = NUM_VARIANTS) -> list[str]:
    """Ask the LLM to produce N alternative phrasings of the question."""
    response = llm.invoke(
        [
            SystemMessage(
                f"Generate exactly {n} alternative phrasings of the user's question. "
                "Each should approach the topic from a different angle or perspective. "
                f"Return only the {n} questions, one per line, no numbering or bullets."
            ),
            HumanMessage(question),
        ]
    )
    raw = response.content.strip().split("\n")
    variants = [q.strip() for q in raw if q.strip()][:n]
    # Always include the original query
    if question not in variants:
        variants = [question] + variants[: n - 1]
    return variants


test_question = "How does RAG Fusion work?"
variants = generate_query_variants(test_question)

print(f"Original: '{test_question}'")
print(f"\nGenerated {len(variants)} variants:")
for i, v in enumerate(variants, 1):
    marker = " (original)" if v == test_question else ""
    print(f"  {i}. {v}{marker}")

In [ ]:
# ─── 3-B: Show how variant diversity affects retrieval ────────────────────────
# Good variants should retrieve DIFFERENT top-1 documents.
# If all variants return identical results, the paraphrases are not diverse enough.

print("Retrieval per variant (top-1 doc only):\n")
retrieved_per_variant = []
for v in variants:
    docs = retriever.invoke(v)
    top_doc = docs[0].page_content if docs else "(empty)"
    retrieved_per_variant.append([d.page_content for d in docs])
    label = (v[:60] + "...") if len(v) > 60 else v
    print(f"  Variant: '{label}'")
    top_label = (top_doc[:80] + "...") if len(top_doc) > 80 else top_doc
    print(f"  Top doc: {top_label}")
    print()

# Count unique documents across all retrievals
all_docs_flat = [doc for ranked_list in retrieved_per_variant for doc in ranked_list]
unique_count = len(set(all_docs_flat))
overlap_count = len(all_docs_flat) - unique_count
print(f"Total retrieved: {len(all_docs_flat)} | Unique: {unique_count} | Overlapping: {overlap_count}")
print("Overlapping docs appear in multiple variant results -- RRF will boost their rank.")

---

## Part 4 — Reciprocal Rank Fusion (RRF)

---

### The RRF Formula

RRF was introduced by Cormack, Clarke, and Buettcher (2009) to combine ranked lists from multiple retrieval systems without needing to normalise relevance scores.

```
RRF_score(doc) = sum_i( 1 / (k + rank_i) )   for i = 1..N

where:
  N      = number of ranked lists (= number of query variants)
  rank_i = position of doc in the i-th ranked list (1-indexed)
  k      = smoothing constant (typically 60)
```

**Why this works:**
- A document that appears at rank 1 in all N lists gets the maximum score: N x 1/(k+1)
- A document that appears in only ONE list at rank 1 gets: 1/(k+1)
- A document consistently ranked 2nd across multiple lists beats a document ranked 1st in one list
- **Cross-list agreement is more reliable than a single high rank**

### Why k=60?

The constant k prevents rank-1 documents from dominating when N is small. With k=60 and N=4:
- Rank 1 in all 4 lists: 4 x 1/61 = 0.0656
- Rank 1 in one list only: 1/61 = 0.0164
- Rank 10 in all 4 lists: 4 x 1/70 = 0.0571

A document at rank 10 across all lists beats a document at rank 1 in only one list — exactly the multi-query consensus signal RAG Fusion is built on.

In [ ]:
# ─── 4-A: RRF implementation from scratch ─────────────────────────────────────


def rrf_merge(all_ranked_lists: list[list[str]], k: int = RRF_K) -> list[str]:
    """
    Reciprocal Rank Fusion across multiple ranked document lists.

    A doc appearing high in MULTIPLE lists gets a high fused score.
    A doc appearing in only one list ranks lower, even if it was rank 1.

    Args:
        all_ranked_lists: list of ranked doc lists, one per query variant
        k: smoothing constant (default 60 from Cormack et al. 2009)

    Returns:
        Deduplicated list of docs sorted by RRF score (highest first)
    """
    scores: dict[str, float] = {}
    for ranked_list in all_ranked_lists:
        for rank, doc in enumerate(ranked_list, start=1):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.__getitem__, reverse=True)


# Toy example: three variants retrieve overlapping documents
toy_lists = [
    ["doc_A", "doc_B", "doc_C"],   # variant-1 ranking
    ["doc_B", "doc_A", "doc_D"],   # variant-2 ranking
    ["doc_A", "doc_D", "doc_E"],   # variant-3 ranking
]

fused = rrf_merge(toy_lists, k=60)
print("Input ranked lists:")
for i, lst in enumerate(toy_lists, 1):
    print(f"  Variant {i}: {lst}")

print("\nRRF scores:")
scores_debug: dict[str, float] = {}
for lst in toy_lists:
    for rank, doc in enumerate(lst, start=1):
        scores_debug[doc] = scores_debug.get(doc, 0.0) + 1.0 / (60 + rank)

for doc, score in sorted(scores_debug.items(), key=lambda x: -x[1]):
    appearances = sum(1 for lst in toy_lists if doc in lst)
    bar = "#" * int(score * 3000)
    print(f"  {doc}: {score:.5f}  {bar}  (in {appearances}/3 lists)")

print(f"\nFused order: {fused}")
print("doc_A wins: rank 1 in two lists + rank 2 in one = most cross-list agreement")

In [ ]:
# ─── 4-B: Visualise RRF score sensitivity to k ────────────────────────────────
# k controls how much rank-1 dominates. Smaller k = rank-1 more impactful.

print("RRF score for rank-1 document in ONE list vs rank-1 in ALL 4 lists:\n")
print(f"{'k':>6}  {'rank-1 (1 list)':>18}  {'rank-1 (4 lists)':>18}  {'ratio':>8}")
print("-" * 60)
for k_val in [10, 30, 60, 100, 200]:
    one_list = 1.0 / (k_val + 1)
    four_lists = 4.0 / (k_val + 1)
    ratio = four_lists / one_list
    print(f"{k_val:>6}  {one_list:>18.5f}  {four_lists:>18.5f}  {ratio:>8.1f}x")

print()
print("The ratio is always 4x (N lists = Nx score) regardless of k.")
print("k controls the ABSOLUTE magnitude -- smaller k amplifies rank differences between docs.")

In [ ]:
# ─── 4-C: Apply RRF to real retrieval results ─────────────────────────────────
# Use the variant results from Part 3 to show a real RRF merge.

fused_docs = rrf_merge(retrieved_per_variant)

print("RRF fusion of real retrieval results:\n")
print("Input ranked lists (one per query variant):")
for i, (v, ranked) in enumerate(zip(variants, retrieved_per_variant), 1):
    label = v[:55] + "..." if len(v) > 55 else v
    print(f"  List {i} [{label}]:")
    for rank, doc in enumerate(ranked, 1):
        doc_label = (doc[:70] + "...") if len(doc) > 70 else doc
        print(f"    Rank {rank}: {doc_label}")

print(f"\nFused ranking ({len(fused_docs)} unique documents):")
for rank, doc in enumerate(fused_docs, 1):
    appearances = sum(1 for lst in retrieved_per_variant if doc in lst)
    score = sum(
        1.0 / (RRF_K + lst.index(doc) + 1)
        for lst in retrieved_per_variant if doc in lst
    )
    print(f"  [{rank}] score={score:.5f} (in {appearances}/{len(variants)} lists)")
    doc_label = (doc[:80] + "...") if len(doc) > 80 else doc
    print(f"       {doc_label}")

---

## Part 5 — Full RAG Fusion Pipeline

---

Now we wire everything into a LangGraph graph. The graph has five nodes:

```
START
  |
  v
generate_variants     LLM generates NUM_VARIANTS paraphrases of the question
  |
  +-- Send(v1) -->
  +-- Send(v2) -->  retrieve_variant  (N parallel workers, one per variant)
  +-- Send(v3) -->
  +-- Send(v4) -->
                    |
                    |  (all workers append to ranked_docs via Annotated[list, add])
                    v
                  fuse       RRF merge of all ranked_docs lists
                    |
                    v
                 generate    LLM answers from fused_docs context
                    |
                    v
                   END
```

### LCEL vs LangGraph Here

We use LangGraph (not a simple LCEL chain) because LCEL does not natively support true parallelism with dynamic fan-out. LangGraph's Send API dispatches N coroutines simultaneously and handles state merging automatically.

In [ ]:
# ─── 5-A: Define state types ───────────────────────────────────────────────────
from operator import add
from typing import Annotated, TypedDict

from langgraph.constants import Send
from langgraph.graph import END, START, StateGraph


class VariantState(TypedDict):
    """State for one parallel retrieval worker."""
    variant: str
    ranked_docs: list[str]


class RAGFusionState(TypedDict):
    """Shared state across the entire RAG Fusion graph."""
    question: str
    variants: list[str]             # populated by generate_variants
    # Annotated[list, add] -- each parallel worker APPENDS one ranked list
    # rather than overwriting. This is the merge primitive.
    ranked_docs: Annotated[list[list[str]], add]
    fused_docs: list[str]           # populated by fuse
    answer: str                     # populated by generate


print("State classes defined.")
print("Key: ranked_docs uses Annotated[list[list[str]], add] for parallel merge.")

In [ ]:
# ─── 5-B: Define graph nodes ───────────────────────────────────────────────────

def node_generate_variants(state: RAGFusionState) -> dict:
    """LLM generates NUM_VARIANTS semantically diverse paraphrases of the original query."""
    response = llm.invoke(
        [
            SystemMessage(
                f"Generate exactly {NUM_VARIANTS} alternative phrasings of the user's question. "
                "Each should approach the topic from a slightly different angle. "
                f"Return only the {NUM_VARIANTS} questions, one per line, no numbering."
            ),
            HumanMessage(state["question"]),
        ]
    )
    raw = response.content.strip().split("\n")
    variants = [q.strip() for q in raw if q.strip()][:NUM_VARIANTS]
    # Always include the original query to ensure it is covered
    if state["question"] not in variants:
        variants = [state["question"]] + variants[: NUM_VARIANTS - 1]
    return {"variants": variants}


def edge_fan_out(state: RAGFusionState) -> list:
    """Return one Send per variant -- LangGraph dispatches all in parallel."""
    return [
        Send("retrieve_variant", {"variant": v, "ranked_docs": []})
        for v in state["variants"]
    ]


def node_retrieve_variant(state: VariantState) -> dict:
    """Retrieve top-K for one query variant. Runs in parallel across all variants."""
    docs = retriever.invoke(state["variant"])
    return {"ranked_docs": [d.page_content for d in docs]}


def node_fuse(state: RAGFusionState) -> dict:
    """Apply RRF to merge all parallel ranked lists into one unified context."""
    return {"fused_docs": rrf_merge(state["ranked_docs"])}


def node_generate(state: RAGFusionState) -> dict:
    """Answer the original question using only the fused context."""
    context = "\n\n".join(state["fused_docs"])
    response = llm.invoke(
        [
            SystemMessage(
                "Answer using only the context below. "
                "The context was retrieved and re-ranked from multiple query paraphrases "
                "for improved recall.\n\nContext:\n" + context
            ),
            HumanMessage(state["question"]),
        ]
    )
    return {"answer": response.content}


print("Nodes defined: generate_variants, fan_out, retrieve_variant, fuse, generate")

In [ ]:
# ─── 5-C: Build and compile the graph ─────────────────────────────────────────

graph = StateGraph(RAGFusionState)

graph.add_node("generate_variants", node_generate_variants)
graph.add_node("retrieve_variant", node_retrieve_variant)
graph.add_node("fuse", node_fuse)
graph.add_node("generate", node_generate)

graph.add_edge(START, "generate_variants")
graph.add_conditional_edges("generate_variants", edge_fan_out, ["retrieve_variant"])
graph.add_edge("retrieve_variant", "fuse")
graph.add_edge("fuse", "generate")
graph.add_edge("generate", END)

app = graph.compile()

print("Graph compiled successfully.")
print("Topology: START -> generate_variants -> [Nx retrieve_variant] -> fuse -> generate -> END")

In [ ]:
# ─── 5-D: Run the full pipeline ────────────────────────────────────────────────

SAMPLE_QUESTIONS = [
    "How does RAG Fusion work?",
    "What is the LangGraph Send API used for?",
    "Explain Reciprocal Rank Fusion for combining search results",
]


def run_rag_fusion(question: str, verbose: bool = True) -> dict:
    """Run the full RAG Fusion pipeline and return the state."""
    result = app.invoke(
        {
            "question": question,
            "variants": [],
            "ranked_docs": [],
            "fused_docs": [],
            "answer": "",
        }
    )
    if verbose:
        print(f"Q: {question}")
        print(f"   Variants generated : {len(result['variants'])}")
        print(f"   Unique fused docs  : {len(result['fused_docs'])}")
        answer_preview = result['answer'][:300]
        print(f"   Answer: {answer_preview}")
        if len(result['answer']) > 300:
            print("   [...]")
        print()
    return result


for q in SAMPLE_QUESTIONS:
    run_rag_fusion(q)

---

## Part 6 — Experiments: Ablations and Sensitivity

---

Now that the full pipeline works, let's understand *how much* each component contributes.

### What to measure

| Parameter | Question | Expected effect |
|-----------|----------|-----------------|
| `NUM_VARIANTS` | More variants = more coverage? | Yes, up to diminishing returns around 4-8 |
| `K` per variant | Wider per-variant retrieval? | More docs but more noise |
| `RRF_K` | Smoothing sensitivity | Low k amplifies rank-1; high k flattens |
| Variant diversity | Paraphrases vs. decompositions | More diverse = more unique retrieval |

The most important experiment is comparing RAG Fusion to single-query RAG on the *same* question — measuring how many more unique relevant chunks fusion recovers.

In [ ]:
# ─── 6-A: RAG Fusion vs single-query RAG -- side by side ──────────────────────

comparison_question = "How does RAG Fusion improve over standard RAG?"

# Single-query baseline
single_query_docs = retriever.invoke(comparison_question)
single_query_set = set(d.page_content for d in single_query_docs)

# RAG Fusion
fusion_result = run_rag_fusion(comparison_question, verbose=False)
fusion_set = set(fusion_result["fused_docs"])

print(f"Question: '{comparison_question}'\n")
print(f"Single-query RAG  : {len(single_query_set)} unique docs")
print(f"RAG Fusion        : {len(fusion_set)} unique docs")
extra = fusion_set - single_query_set
print(f"New docs from fusion: {len(extra)} docs not in single-query result")
print()

if extra:
    print("Additional docs recovered by RAG Fusion (not returned by single query):")
    for doc in extra:
        doc_label = (doc[:90] + "...") if len(doc) > 90 else doc
        print(f"  -> {doc_label}")
else:
    print("(On this query, single-query RAG happened to retrieve the same top docs.)")
    print("Try more obscure or multi-faceted questions to see the fusion benefit.")

In [ ]:
# ─── 6-B: Effect of NUM_VARIANTS on unique doc coverage ──────────────────────


def fusion_coverage(question: str, n_variants: int) -> dict:
    """Build a quick inline pipeline to test different NUM_VARIANTS values."""
    these_variants = generate_query_variants(question, n=n_variants)
    all_ranked = []
    for v in these_variants:
        docs = retriever.invoke(v)
        all_ranked.append([d.page_content for d in docs])
    fused = rrf_merge(all_ranked)
    return {
        "n_variants": n_variants,
        "unique_docs": len(fused),
        "total_retrieved": sum(len(lst) for lst in all_ranked),
    }


ablation_question = "What is Reciprocal Rank Fusion and why is it used?"
print(f"Ablation: NUM_VARIANTS effect on coverage")
print(f"Question: '{ablation_question}'\n")
print(f"{'N variants':>12}  {'Unique docs':>12}  {'Total retrieved':>16}")
print("-" * 45)

for n in [1, 2, 4, 6, 8]:
    stats = fusion_coverage(ablation_question, n_variants=n)
    label = "(baseline)" if n == 1 else ""
    print(f"{n:>12}  {stats['unique_docs']:>12}  {stats['total_retrieved']:>16}  {label}")

---

## Exercises

---

### Exercise 1 — Inspect Variant Quality

Run `generate_query_variants()` on three different questions — one technical, one ambiguous, one multi-hop. Do the variants genuinely approach the topic from different angles? What happens if the original question is already very specific?

**Bonus:** Try a temperature of 0.7 vs 0.0 in the LLM call. Does higher temperature produce more diverse variants?

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
questions_to_test = [
    "TODO: a technical question (e.g. about RRF or LangGraph)",
    "TODO: an ambiguous question (e.g. 'what is a good retrieval method?')",
    "TODO: a multi-hop question requiring two reasoning steps",
]

for q in questions_to_test:
    if q.startswith("TODO"):
        continue
    print(f"Question: '{q}'")
    these_variants = generate_query_variants(q, n=4)
    for i, v in enumerate(these_variants, 1):
        print(f"  {i}. {v}")
    print()

### Exercise 2 — Compare Fusion vs Single-Query on Your Own KB

Add 5 or more documents on a topic of your choice to the knowledge base. Run both standard retrieval and RAG Fusion. For which questions does fusion recover more relevant docs?

**What to observe:** Fusion wins most when (a) the question has vocabulary that differs from document vocabulary, or (b) relevant information is spread across multiple chunks that no single query captures.

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────
my_extra_docs = [
    Document(page_content="TODO: replace with your own content"),
    # Add more documents here...
]

# TODO: add to the vectorstore and create a new retriever
# vectorstore.add_documents(my_extra_docs)

my_question = "TODO: pick a question that tests vocabulary mismatch"

# TODO: run both retrieval strategies and compare
# single_result = retriever.invoke(my_question)
# fusion_result = run_rag_fusion(my_question)
# Compare: set(d.page_content for d in single_result) vs set(fusion_result['fused_docs'])

### Exercise 3 — RRF_K Sensitivity

Change `RRF_K` to 10, 30, 60, and 100 in the `rrf_merge` function (or pass it directly). Use the toy ranked lists from cell 4-A. How does the *relative* ranking of documents change? At what value does rank-1 stop dominating?

**Expected finding:** The final order rarely changes (RRF is robust to k), but score magnitude scales with k. The paper recommends 60 because it produces stable rankings across diverse retrieval systems.

In [ ]:
# ── Exercise 3 Starter ────────────────────────────────────────────────────────
toy_lists_ex3 = [
    ["doc_A", "doc_B", "doc_C"],
    ["doc_B", "doc_A", "doc_D"],
    ["doc_C", "doc_D", "doc_A"],
]

# TODO: run rrf_merge with k=10, 30, 60, 100 and compare score magnitudes
for k_val in [10, 30, 60, 100]:
    fused = rrf_merge(toy_lists_ex3, k=k_val)
    # TODO: add per-doc scores to see the magnitude difference
    print(f"k={k_val:>3}: {fused}")

---

## Answer Key

---

Use this section after attempting the exercises on your own.

### Exercise 1 — Variant Quality

**What good variants look like:** Each paraphrase targets a genuinely different aspect of the question. For *"How does RAG Fusion work?"*, good variants might ask about the RRF algorithm, about the Send API pattern, about how variants are generated, and about how it compares to standard RAG.

**Very specific questions** (e.g., *"What is the exact RRF formula?"*) produce paraphrases that are all very similar — there are not many angles. Fusion helps less here and costs the same. Save it for broad or ambiguous questions.

**Temperature 0.7 vs 0.0:** Higher temperature typically produces more diverse paraphrases (useful for fusion) but less reliably formatted output (may need more parsing). A common pattern is to generate with temperature=0.7 and validate format with a post-processing step.

In [ ]:
# Answer key -- Exercise 1: sample output for a technical question
sample_technical_q = "What is the LangGraph Send API?"
sample_variants = generate_query_variants(sample_technical_q, n=4)

print(f"Original: '{sample_technical_q}'")
print(f"\n{len(sample_variants)} variants generated:")
for i, v in enumerate(sample_variants, 1):
    print(f"  {i}. {v}")

print()
print("A strong variant set covers different aspects:")
print("  - What it does (definition)")
print("  - How to use it (usage pattern)")
print("  - Why it exists (motivation / problem it solves)")
print("  - Comparison (how it differs from standard edges)")

### Exercise 2 — Fusion vs Single-Query

**Sample answer structure:** Create a small KB with 8-10 documents on a topic. Pick a question where the vocabulary in the question differs from the vocabulary in the documents (e.g., the document says *"adverse cardiac events"* but you ask *"heart attack risk"*).

**What to look for:** Single-query RAG returns the chunks whose text embedding is closest to your query embedding. RAG Fusion returns the union of what N different phrasings retrieve — typically 2-4 more unique documents, any of which may contain the specific detail the user needed.

**When fusion does not help:** If the document vocabulary already closely matches how users ask questions (e.g., a FAQ document written to match user queries), fusion adds cost without much benefit.

### Exercise 3 — RRF_K Sensitivity

**Sample answer:** With the toy lists `[A,B,C], [B,A,D], [C,D,A]`, doc_A appears in all three lists (ranks 1, 2, 3) and doc_B and doc_C appear in two. The final RRF ranking should be A > B or C > D regardless of k.

What changes with k is the score *magnitude* — not the order. The original paper chose k=60 empirically across many retrieval benchmarks as a stable default that prevents any single high-ranked document from dominating when only one of N variants retrieves it.

In [ ]:
# Answer key -- Exercise 3: full score breakdown per k value
toy_lists_answer = [
    ["doc_A", "doc_B", "doc_C"],
    ["doc_B", "doc_A", "doc_D"],
    ["doc_C", "doc_D", "doc_A"],
]

for k_val in [10, 30, 60, 100]:
    doc_scores: dict[str, float] = {}
    for lst in toy_lists_answer:
        for rank, doc in enumerate(lst, start=1):
            doc_scores[doc] = doc_scores.get(doc, 0.0) + 1.0 / (k_val + rank)
    ordered = sorted(doc_scores, key=doc_scores.__getitem__, reverse=True)
    print(f"k={k_val:>3}: ", end="")
    for doc in ordered:
        print(f"{doc}={doc_scores[doc]:.5f}  ", end="")
    print()

print()
print("Observation: doc_A wins at every k value.")
print("k controls score magnitude, not the final ranking order.")

---

## What's Next?

You now understand RAG Fusion end-to-end — from vocabulary mismatch diagnosis, through parallel fan-out, to RRF scoring. Here is where to go deeper:

### Continue the retrieval series
- **Example 27 — Self-RAG** (`../27-self-rag/`): decide *whether* to retrieve at all using LLM reflection gates before and after retrieval
- **Example 28 — Parallel Subgraph** (`../28-parallel-subgraph/`): the LangGraph Send API map-reduce pattern in isolation — document summarisation as a worked example
- **Example 25 — Adaptive RAG** (`../25-adaptive-rag/`): route queries to the right retrieval strategy (local vs web vs no-retrieval) before running anything
- **Example 22 — Hybrid Search RAG** (`../22-hybrid-search-rag/`): ensemble BM25 + vector search with a single query (complementary to fusion)

### Improve the system
- **Add diversity constraints:** Use MMR (Maximum Marginal Relevance) within each variant retrieval to reduce redundancy before fusion
- **Use HyDE instead of paraphrase variants:** Generate a hypothetical answer, embed it, retrieve based on the answer's embedding space
- **Score-weighted fusion:** If your vector store returns similarity scores, combine them with RRF scores for a hybrid ranking signal
- **Evaluate:** See **Example 16 — RAG Evaluation** for automatic RAGAS scoring — measure Faithfulness and Context Recall before and after fusion

### Academic references

> Cormack, G. V., Clarke, C. L. A., & Buettcher, S. (2009). *Reciprocal Rank Fusion outperforms Condorcet and individual Rank Learning Methods.* SIGIR 2009. https://dl.acm.org/doi/10.1145/1571941.1572114
>
> Lewis, P., Perez, E., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401
>
> Ma, X., et al. (2023). *Query Rewriting for Retrieval-Augmented Large Language Models.* EMNLP 2023. https://arxiv.org/abs/2305.14283
>
> Gao, Y., Xiong, Y., et al. (2023). *Retrieval-Augmented Generation for Large Language Models: A Survey.* https://arxiv.org/abs/2312.10997
>
> Lawrence, S. (2024). *RAG-Fusion.* GitHub. https://github.com/Raudaschl/rag-fusion

---

*Workshop complete. Next step: example 27 (Self-RAG) or 25 (Adaptive RAG) to add query-level intelligence on top of what you just built.*